In [ ]:
from playwright.sync_api import Playwright, sync_playwright


def run(playwright: Playwright) -> None:
    browser = playwright.chromium.launch(headless=False)
    context = browser.new_context()
    page = context.new_page()

    page.goto("https://tmailor.com/")

    # 1️⃣ Wait until email address is generated
    page.wait_for_function(
        """() => {
            const el = document.querySelector('input[name="currentEmailAddress"]');
            return el && el.value && el.value.includes('@');
        }"""
    )
    print("Email address ready")

    # 2️⃣ Wait for inbox message & click it
    inbox_email = page.locator(
        'a.temp-subject:has-text("verify your email")'
    ).first

    inbox_email.wait_for(timeout=120_000)
    inbox_email.click()
    print("Inbox email opened")

    # 3️⃣ The email body is usually inside an iframe
    frame = page.frame_locator("iframe")

    # 4️⃣ Wait for Verify button inside email
    verify_button = frame.locator(
        'a:has-text("Verify my email")'
    )

    verify_button.wait_for(timeout=30_000)

    # 5️⃣ Click and capture new tab
    with context.expect_page() as new_page_info:
        verify_button.click()

    verify_page = new_page_info.value
    verify_page.wait_for_load_state()

    print("Verification page opened:", verify_page.url)

    input("Press Enter to close...")
    browser.close()


with sync_playwright() as playwright:
    run(playwright)
